# Esperimenti tesi

Il primo esperimento che voglio fare è fittare una serie di alberi con diverse profondità su dei dati sintetici $y \in \{1,-1\}$ con loss $\ell$ e verificare se la differenza tra l'errore su un training set ($\hat R$) e l'errore su un test set ($R$) segue l'andamento previsto dal bound teorico
\begin{equation}
0 \le \mathbb{E}_D[R(\hat f_D)-\hat R(\hat f_D)] \le {2 M} \sqrt{\frac{\log \vert \mathcal{H}\vert}{2n}}
\end{equation}
dove $\ell(\cdot) \le M$, cioè aumenta all'aumentare della profondità del mio albero in un modo controllato. 

N.B. parlo di profondità perché nel caso degli alberi, la cardinalità dell'insieme di ipotesi $\mathcal{H}$ è proporzionale alla profondità, in altre parole più profondo l'albero più comlpesso il modello che impariamo. 

### dati

Costruiamo un dataset sintetico di size $n$, secondo la legge 
\begin{gather}
x \sim \mathrm{Unif}(\text{supp}(0,10))\tag{2}\\
y \coloneqq \mathrm{sign}(f(x) + \varepsilon), \quad \varepsilon \sim \mathcal{N}(0,\sigma^2)\tag{3}
\end{gather}
for some $f: X \to Y$ convex and where $\text{supp}(0,10)$ is a discretization of the interval $[0,10]$ into $N$ points.  

we want 
\begin{equation}
n_\text{train} \ll N \ll n_\text{test} \approx \infty
\end{equation}

### modelli e plot
we train **binary trees** with depths ranging from 1 to a multiple of the $\log_2 n$ threshold to see long term trends, using a **counting loss**. 

we compute the expected values of $R$ and $\hat R$ (and their difference) over the disorder of the data $D$ averaging over a number of training samples equal to ```num_iter```. 

we then do a number of plots to visualize to what extent empirical results confirm the theoretical bound $\text{(1)}$. 


In [6]:
print('mare')

mare


### build dataset




In [ ]:
# build dataset

import numpy as np

rng = np.random.default_rng(seed=4)  

def sample_x(n, N, x_measure = "uniform", l=0.0, u=10.0, rng=rng):
    """sample n points from supp evenly spaced points on [l, u]"""
    support = np.linspace(l, u, N)
    if x_measure == "uniform":
        return rng.choice(support, size=n, replace=True)
    else:
        print('specify a measure over X')

def sample_y(x, f, sigma=1.0, rng=rng):
    """y = sign(f(x) + e), e ~ N(0, sigma)"""
    e = sigma * rng.standard_normal(size=len(x))
    return np.where(f(x) + e >= 0, 1, -1) # include lo 0 come classe 1 per evitare che l'albero crei un'altra classe

def sample_y_bernoulli(x, f, p=0.8, rng=rng):
    """y = sign(f(x)) * b, b = +1 w.p. p, b = -1 w.p. 1-p"""
    b = rng.choice(np.array([1, -1]), size=len(x), p=[p, 1 - p])
    return np.where(f(x) >= 0, 1, -1) * b


### explore dataset

In [ ]:

# explore dataset
import matplotlib.pyplot as plt

def explore_data(f, n, N, x_measure = "uniform", l=0.0, u=10.0, sigmas=(10**(-4), 1, 5, 10),
                  n_grid=20_000, error_type='gaussian', probs=(0.5, 0.7, 0.9, 0.99), axes=None):
    '''
    plot the data and a sample training set to see where the points are 
    and how the noise influences the distribution.

    error_type selects which noise model to sweep over the 4 panels:
    'gaussian' sweeps sigmas (points scattered at f(x)+e, so the noise's
    effect on the position is visible); 'bernoulli' sweeps probs (there's no
    continuous noisy value here, just a sign flip, so points are scattered
    on the f(x) curve itself and only their color shows the flip).

    points are colored by whether the sampled y agrees with the noiseless
    sign(f(x)) — blue for correctly classified, red for misclassified — and
    each panel title also reports the misclassified count, a direct read of
    how much the noise level actually flips labels.
    '''
    x_plot = np.linspace(l, u, n_grid)

    # same x locations across panels so only the noise differs
    x = sample_x(n=n, N=N, x_measure = x_measure, l=l, u=u)
    true_y = np.where(f(x) >= 0, 1, -1)

    # x-values where f crosses the decision boundary (f(x) = 0), found by
    # linear interpolation between sign changes of f on the dense grid x_plot
    f_plot = f(x_plot)
    crossing_idx = np.where(np.diff(np.sign(f_plot)) != 0)[0]
    x0, x1 = x_plot[crossing_idx], x_plot[crossing_idx + 1]
    y0, y1 = f_plot[crossing_idx], f_plot[crossing_idx + 1]
    roots = x0 - y0 * (x1 - x0) / (y1 - y0)

    params = sigmas if error_type == 'gaussian' else probs
    if axes is None:
        fig, axs = plt.subplots(2, 2, figsize=(11, 9), sharex=True, sharey=True)
        if error_type == 'gaussian':
            fig.suptitle(r'data distribution for different noise levels $\sigma$', fontsize=13)
        elif error_type == 'bernoulli':
            fig.suptitle(r'data distribution for different flip probabilities $p$', fontsize=13)
    else:
        axs = np.asarray(axes)

    for ax, param in zip(axs.flat, params):
        if error_type == 'gaussian':
            e = param * rng.standard_normal(size=len(x))
            noisy_f = f(x) + e
            y = np.where(noisy_f >= 0, 1, -1)
            panel_title = rf'$\sigma$ = {param}'
        elif error_type == 'bernoulli':
            y = sample_y_bernoulli(x, f, p=param)
            noisy_f = f(x)
            panel_title = rf'$p$ = {param}'

        misclassified = y != true_y
        n_misclassified = np.sum(misclassified)
        panel_title += f'  ({n_misclassified}/{n} misclassified)'

        ax.plot(x_plot, f(x_plot), color='#0b0b0b', linewidth=1, label=r'$f(x)$')
        ax.axhline(0, color='#c3c2b7', linestyle='--', linewidth=1.2, label='decision boundary')
        for i, root in enumerate(roots):
            ax.axvline(root, color='#898781', linestyle=':', linewidth=1.5,
                       label=r'$f(x)=0$' if i == 0 else None)
        ax.scatter(x[~misclassified], noisy_f[~misclassified], color='#2a78d6', s=20, alpha=0.8, label='correctly classified')
        ax.scatter(x[misclassified],  noisy_f[misclassified],  color='#e34948', s=20, alpha=0.8, label='misclassified')

        ax.set_title(panel_title)
        # x ticks at the crossings of f with the x axis
        ax.set_xticks(roots)
        ax.set_xticklabels([f'{r:.2f}' for r in roots], rotation=45)
        ax.set_xlabel('x')
        ax.set_ylabel(r'$f(x)+\varepsilon$' if error_type == 'gaussian' else r'$f(x)$')
        ax.grid(True, alpha=0.3)

    axs.flat[0].legend(loc='upper center')
    if axes is None:
        plt.tight_layout(rect=[0, 0, 1, 0.96])
        plt.show()

def find_sigma_star(f, p, n, N, sigmas=np.arange(1, 10.01, 0.25), num_samples=100, plot=True):
    '''
    calibrate gaussian noise against bernoulli noise: for each sigma in
    sigmas, average over num_samples training sets of size n the fraction of
    labels that disagree with the noiseless sign(f(x)), and return the
    sigma whose average is closest to the bernoulli one, 1 - p.

    the fraction flipped by gaussian noise depends on the shape of f (how
    much of the domain sits close to f = 0), so sigma_star has to be
    recomputed for every f, while for bernoulli noise it is always 1 - p.
    '''
    frac_avg = []
    for sigma in sigmas:
        frac = []
        for _ in range(num_samples):
            x = sample_x(n, N)
            true_y = np.where(f(x) >= 0, 1, -1)
            frac.append(np.mean(sample_y(x, f, sigma=sigma) != true_y))
        frac_avg.append(np.mean(frac))
    frac_avg = np.array(frac_avg)
    sigma_star = sigmas[np.argmin(np.abs(frac_avg - (1 - p)))]

    if plot:
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.plot(sigmas, frac_avg, color='#00CED1', marker='o', label=r'avg fraction misclassified, gaussian')
        ax.axhline(1 - p, color='gray', linestyle='--', linewidth=1.2, label=rf'$1 - p$ = {1 - p:.2f}')
        ax.axvline(sigma_star, color='red', linestyle='--', linewidth=1.2, label=rf'$\sigma^*$ = {sigma_star}')
        ax.set_title(f'misclassified inputs against noise level (avg over {num_samples} samples of size {n})')
        ax.set_xlabel(r'$\sigma$'); ax.set_ylabel('fraction misclassified')
        ax.legend(); ax.grid(True, alpha=0.3)
        plt.show()
    return sigma_star

### preliminaries

In [ ]:

# almost-always-fixed details

def get_model_class(model):
    """works for both a class (trees) and a functools.partial (e.g. MLP)"""
    return model.func if hasattr(model, 'func') else model

def get_depths_list(t, model):
    model_class = get_model_class(model)
    if model_class == DecisionTreeClassifier:
        depths = np.arange(1, int(np.floor(5 * t)) + 1)
        depths_float = depths.astype(float)
    elif model_class == MLPClassifier:
        max_pow = int(np.ceil(np.log2(5 * t)))
        depths = np.array([2**i for i in range(max_pow + 1)], dtype=float)
        depths_float = depths
    elif model_class == RandomForestClassifier:
        # hybrid complexity measure (Belkin et al.'s random forest double-
        # descent experiment): first grow a single tree (n_estimators=1)
        # deeper and deeper until it's complex enough to interpolate, then
        # freeze the tree fully grown (max_depth=None) and instead grow the
        # ensemble by adding more trees. there's no overfitting-threshold
        # scale for this hybrid measure, so t is unused and these ranges are
        # fixed rather than derived from n.
        max_single_tree_depth = 20
        max_n_estimators = 200

        depth_phase        = np.arange(1, max_single_tree_depth + 1)
        n_estimators_phase = np.arange(2, max_n_estimators + 1)

        # column 0: max_depth to use (-1 is a sentinel for "None", i.e. fully
        # grown/interpolating); column 1: n_estimators to use
        phase_one = np.column_stack([depth_phase, np.ones_like(depth_phase)])
        phase_two = np.column_stack([-np.ones_like(n_estimators_phase), n_estimators_phase])
        depths = np.concatenate([phase_one, phase_two], axis=0)

        # depth and n_estimators live on different scales, so a plain
        # increasing index is used as the x-axis / complexity value instead
        depths_float = np.arange(1, len(depths) + 1, dtype=float)
    return depths, depths_float

def centered_parabola(x):
    return (x-5)**2 - 6.25

# define some sine functions with fixed height at 1.2 and given number of crossings in the interval [0,10 ]
def sine(x, m, u = 10.0):
    # obtain a number of crossings equal to the given one, in the interval [0, u]
    c = (np.pi * (m +1)) / u
    return 5*np.sin(c*x)

### define bounds

#### compute $\inf R^*_d$

In [ ]:
# predict R^*_d, the minimum risk attainable by trees of depth at most d

import numpy as np

def dp_R_star_d(f, depth, N=10_000, l=0.0, u=10.0, p=0.8):
    """
    Computes exactly
        inf_{h in H_d} R(h)

    Come leggerlo: dalla fine, altrimenti non capisci una minchia

    cost[i, j]
      è una matrice che ha i blocchi sulle rows e la mia predizione sulle colonne, e mi dà il costo di predire j per il blocco i
      cost[i, 0] = costo se al blocco contiguo i predico -1
      cost[i, 1] = costo se al blocco contiguo i predico 1

    dp[k, j]
      è una matrice che ha il numero di split del mio albero sulle rows e la mia ULTIMA predizione sulle colonne
    """

    depth = np.asarray(depth, dtype=int)

    # exact support of X
    x = np.linspace(l, u, N)
    true_labels = np.where(f(x) >= 0, 1, -1)

    # prima calcolo gli indici dei blocchi contigui di 1 o -1
    change_points = np.where(np.diff(true_labels) != 0)[0] + 1
    starts = np.r_[0, change_points]

    run_labels = true_labels[starts]

    # poi qui in un modo mega clever mi calcolo i pesi di ogni blocco contiguo (quante x ha dentro)
    ends = np.r_[change_points, N]
    run_weights = (ends - starts) / N

    # questo serve per sapere su quanti blocchi devo poi runnare dp
    M = len(run_labels)

    # BO
    max_K = min(2 ** depth.max(), M)

    # qui setto che all'indice 0 di labels ho -1, all'indice 1 ho 1
    labels = np.array([-1, 1])

    # qui calcolo l'expected error contributed by each block
    # cost diventa una matrice che si legge come cost[i, 0] è il costo se al blocco i
    # predico -1, cost[i, 1] è il costo se al blocco i predico 1
    cost = np.empty((M, 2))

    for i in range(M):
        for j, pred in enumerate(labels):
            if pred == run_labels[i]:
                cost[i, j] = run_weights[i] * (1 - p)
            else:
                cost[i, j] = run_weights[i] * p

    # qui runno dp. dp è una matrice che si legge come dp[k, j] è il rischio con al max k intervalli dove nell'ultimo blocco ho predetto j. 
    # dp E' UNA MATRICE CHE VIENE AGGIORNATA AD OGNI LOOP i in range(1, M) PERCHE' SOMMO I CONTRIBUTI PER OGNI BLOCCO 
    # dp[k, j]:
    # minimum risk so far using exactly k intervals,
    # with prediction labels[j] on the last interval

    # inizializzo matrice con dim max numero di intervalli ever X 2, e tutte le celle a infinito
    dp = np.full((max_K + 1, 2), np.inf)

    # poi guardo i blocchi uno ad uno e aggiorno la mia matrice dp di conseguenza

    # sul primo blocco posso usare solo un intervallo, quindi dico che usando un intervallo ho il costo sul primo blocco (0) di prevedere : (che è -1 o 1)
    dp[1, :] = cost[0, :]

    # su ognuno dei blocchi dopo, 
    for i in range(1, M):
        # creo una matrice uguale a quella di prima con tutti valori \infty. questa viene risetatta per ogni i, perché mi serve intonsa all'inizio di ogni i. i valori che calcolo di volta in volta, alla fine del loop sono salvati in dp, quindi passano a same e switch, e quindi a new = cost + min(same, switch), che poi li ripassa a dp 
        new = np.full_like(dp, np.inf)

        # il max numero di intervalli che posso fare è il minimo tra max_K e il numero di blocchi visti finora + 1
        max_k_here = min(i + 1, max_K)

        # per ogni numero di intervalli k voglio aggiornare new con il costo con al massimo k intervalli per blocco i 
        for k in range(1, max_k_here + 1):
            for j in range(2):
                # per ogni possibile predizione j = 1 o j = -1,

                # iter i = 1: same è il costo di, con 1 intervallo, predire ancora j 
                # iter i = 2, k = 2: same è il costo di mantenere la stessa label, infatti va a prendere il rischio con 2 intervalli di j, che era infinito
                # iter i = 3, k = 1: same è il costo di fare solo un intervallo, quindi prende il costo di 2 intervalli calcolato prima
                same = dp[k, j]

                # iter 1: switch è infinito
                # iter 2: switch è il costo di cambiare, e quindi va a prendere il costo con k = 1 di cambiare label, che è cost[0, 1-j]
                switch = dp[k - 1, 1 - j] if k > 1 else np.inf

                # iter 1: la cella (k, j) di new diventa il costo che ho di predire j a quel blocco + il minimo tra same e switch, che sarà same
                # iter 2: la cella (2, j) di new diventa il costo di predire j in questo blocco più switch, perché so che mi conviene cambiare 
                new[k, j] = cost[i, j] + min(same, switch) 

        # alla fine del k loop, setto dp uguale alla new che ho ottenuto
        # iter 1: dp diventa new. iter i = 1 finisce perché non posso fare più intervalli. saltiamo iter i = 2, k = 1 perché è uguale.
        dp = new

    # exact_k_risk è una lista lunga max_K che per ogni k prende il minimo tra le due colonne, cioè tra che l'ultima predizione sia 1 o -1
    exact_k_risk = np.min(dp, axis=1)
    # at_most_k_risk prende il minimo tra le due colonne tra tutte le rows up to k (cioe tra 2k numeri prendi il minimo perché selezioni l'albero con al max k intervalli che minimizza il rischio)
    at_most_k_risk = np.minimum.accumulate(exact_k_risk[1:])

    risks = []

    for d in depth:
        # per ogni depth, prendi l'albero che minimizza il minimo rischio tra gli alberi con al massimo 2^d o M leaves/intervalli/blocchi CHIAMALI COME VUOI (non ne fanno ami più di M)
        K = min(2 ** d, M)
        risks.append(at_most_k_risk[K - 1])

    return np.asarray(risks)

### bounds

The ```noise_approxerr_bound``` bound is

\begin{equation}

 \min\left\{ 
\sqrt{\frac{(A_{\mathcal H}+\sigma^2)\log \vert\mathcal H \vert}{n}} + \frac{\log\vert\mathcal H \vert}{n}, 
\sqrt{A_{\mathcal H}+\sigma^2} + 1 \right\}

\end{equation}

In [ ]:

# define bounds! 

from scipy.optimize import brentq
from scipy.integrate import quad
import numpy as np

rng = np.random.default_rng(seed=4)  

def log_card_H(depth, N= 10_000):
    return depth

def union_bound(depth, n, N = 10_000):
    # Thm 2.5 o inequality (43), a meno di costanti
    return np.sqrt(log_card_H(depth)/ n)

def radem_massart_bound(depth, n, N = 10_000):
    return min(np.sqrt(log_card_H(depth)/ n), 0.5)

def noise_approxerr_bound(depth, n, R_star_d, N = 10_000, p = 0.8):
    A = (R_star_d*log_card_H(depth))/ n
    B = np.sqrt(A) + log_card_H(depth)/ n
    C = np.sqrt(R_star_d) 
    return min(B, C)

BOUND_REGISTRY = {
    "unionbound": union_bound,
    "rademmassart": radem_massart_bound,
    "noiseapproxerr": noise_approxerr_bound
}

# formula mostrata in legenda per ogni bound (forma del bound, a meno di costanti)
BOUND_LABELS = {
    "unionbound": r'$\sqrt{d/n}$',
    "rademmassart": r'$\min\{\sqrt{d/n},1\}$',
    "noiseapproxerr" : r'$\min\{\sqrt{(R^*_{\mathscr{H}_d} d)/n} + d/n, \sqrt{R^*_{\mathscr{H}_d}}+1\}$'
}

### build model and create plots 

In [ ]:
# build model and create plots

from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

def main_loop(x_test, y_test, n, T, model, sigma = 5, x_measure = "uniform", N=10_000,
              l = 0.0, u = 10.0, f=centered_parabola, noise_type = 'gaussian', p = 0.8):
    # noise_par
    '''
    function to build models and obtain the exp values of R, R_hat, and their difference for a number of depths.

    sample T training sets once, then for each depth:
    fit model to each of those same T training sets;
    compute the metrics;
    average at the end to cancel the disorder over the data randomness.

    reusing the same T datasets at every depth (instead of drawing
    T fresh ones per depth) means depth is the only thing that changes
    across the curve — the disorder D is common across depths, so depth-to-
    depth differences aren't also mixed with independent sampling noise.

    also keeps, for each depth, the fitted model (among the T) that
    minimizes the training error hat_R, so it can be plotted/animated later
    as "the" learned function for that depth.

    return lists of the 3 metrics, one value for each depth, plus the best
    prediction/sample per depth and the x_grid they were evaluated on.
    '''

    # used for plotting best prediction
    x_grid = np.linspace(l, u, N)

    # overfitting threshold: left as None for models with no threshold logic
    # defined here (e.g. RandomForestClassifier's hybrid complexity measure),
    # so the plots simply skip drawing it
    threshold = None
    if get_model_class(model) == DecisionTreeClassifier:
        threshold = np.log2(n)
    # error_par
    elif get_model_class(model) == MLPClassifier:
        # ~3 params per hidden unit with 1D input → threshold at n/3
        threshold = n / 3

    depths_list, _ = get_depths_list(threshold, model)

    # sample the T training sets once: reused, unchanged, at every depth
    xsamples = []; ysamples = []
    for i in range(T):
        x = sample_x(n, N, x_measure = x_measure, l = l, u = u)
        if noise_type == 'gaussian':
            y = sample_y(x, f, sigma)
        elif noise_type == 'bernoulli':
            y = sample_y_bernoulli(x, f, p = p)
        xsamples.append(x); ysamples.append(y)

    E_hat_Rs = []; E_Rs = []; E_gaps = []
    std_hat_Rs = []; std_Rs = []; std_gaps = []
    best_predictions = []; best_xsample = []; best_ysample = []
    for d in depths_list:
        # initialize for each depth the list of R hats, Rs, and gaps (will do mean) 
        hat_Rs = []; Rs = []; gaps = []; 
        # and the quantities for plotting best function (will do argmin)
        predictions = []
        for i in range(T):
            x, y = xsamples[i], ysamples[i]
            # fit model to training data
            if get_model_class(model) == DecisionTreeClassifier:
                clf = model(max_depth=int(d), random_state=42)
            elif get_model_class(model) == MLPClassifier:
                clf = model(hidden_layer_sizes=(int(d),))
            elif get_model_class(model) == RandomForestClassifier:
                # d is a (max_depth_or_sentinel, n_estimators) row from
                # get_depths_list's hybrid measure: -1 means "fully grown"
                depth_val, n_estimators_val = int(d[0]), int(d[1])
                max_depth = None if depth_val == -1 else depth_val
                clf = model(max_depth=max_depth, n_estimators=n_estimators_val, random_state=42)
            clf.fit(x.reshape(-1, 1), y)
            predictions.append(clf.predict(x_grid.reshape(-1, 1)))

            # compute errors and gap
            train_err = 1 - clf.score(x.reshape(-1, 1), y)
            test_err  = 1 - clf.score(x_test.reshape(-1, 1), y_test)
            gap = test_err - train_err
            hat_Rs.append(train_err); Rs.append(test_err); gaps.append(gap)

        # store values for plotting best learned function
        best_idx = 1
        # int(np.argmin(hat_Rs))
        best_predictions.append(predictions[best_idx])
        best_xsample.append(xsamples[best_idx])
        best_ysample.append(ysamples[best_idx])

        # out of the T loop, we append the avg value for the current depth d
        E_hat_Rs.append(np.mean(hat_Rs))
        E_Rs.append(np.mean(Rs))
        E_gaps.append(np.mean(gaps))
        # and the std across the T samples, for the variance bands in plots()
        std_hat_Rs.append(np.std(hat_Rs))
        std_Rs.append(np.std(Rs))
        std_gaps.append(np.std(gaps))

        # print(f'Depth: ', d, 'R cappuccios: ', hat_Rs)
    return (np.array(E_hat_Rs), np.array(E_Rs), np.array(E_gaps),
            np.array(std_hat_Rs), np.array(std_Rs), np.array(std_gaps),
            best_predictions, best_xsample, best_ysample, x_grid)


def plots(E_hat_Rs, E_Rs, E_gaps,
          # variables for visualizations in the plots
          n, T,
          # variable for obtaining useful quantities
          model, sigma = 5,
          # std across the T samples (as returned by main_loop()), used
          # to draw a shaded band of ± 1 std around the corresponding curve;
          # left as None to skip the band, e.g. when plotting from data that
          # doesn't carry a std (older call sites, or axes reused across runs),
          bound_name = "naive", bound_params={},
          std_hat_Rs=None, std_Rs=None, std_gaps=None,
          one=True, two=True, three=True,
          axes=None, noise_type = 'gaussian', p = 0.8, N = 10_000,
          # bound_name selects the theoretical bound function from BOUND_REGISTRY; 
          # bound_params contains its extra arguments oltre a n e le depths
          # (e.g. inf_R for "variance_inftesterror", Rhat for "variance_trainerror")
          ):

    '''
    function to obtain the following plots, given the metrics obtained with main_loop():

    1) classical train vs. test error plot, against depth of the model;
    2) plots the empirical gap and the theoretical function that should upper bound the gap depending on the depth;
    3) plots the same as (2) with as RHS E[hat_R] + bound.

    each curve coming from main_loop's T samples (E_hat_Rs, E_Rs,
    E_gaps) can be shown with a shaded ± 1 std band behind it, in a lighter
    shade of the same color, if the corresponding std array is passed.
    '''
    
    # depths and quantities for plotting overft threshold and the bound function.
    # threshold is left as None for models with no threshold logic defined
    # here (e.g. RandomForestClassifier's hybrid complexity measure), so the
    # "if threshold:" checks below simply skip drawing it
    threshold = None
    if get_model_class(model) == DecisionTreeClassifier:
        threshold = np.log2(n)

    elif get_model_class(model) == MLPClassifier:
        # ~3 params per hidden unit with 1D input → threshold at n/3
        threshold = n / 3

    _, depths_list_float = get_depths_list(threshold, model)
    cardinalities = [(2**d)*np.log(2*N) for d in depths_list_float]

    bound_fn = BOUND_REGISTRY[bound_name]
    bound_label = BOUND_LABELS.get(bound_name, bound_name)
    ref_label = r'$E[\hat R]$'
    gap_label = r'$E[R - \hat R]$'
    print(f'[checkpoint] bound: "{bound_name}" -> {bound_fn.__name__}, params: {list(bound_params)}')

    # np.vectorize calls bound_fn once per depth, broadcasting every argument:
    # scalars in bound_params are shared across depths, arrays with one value
    # per depth (e.g. Rhat = E_hat_Rs) are read element by element
    bound = np.vectorize(bound_fn)(depths_list_float, n, **bound_params)

    # cluade logic for getting multiple plots at a time
    n_plots = sum([one, two, three])
    if n_plots == 0:
        return
    if axes is None:
        fig, axs = plt.subplots(n_plots, 1, figsize=(10, 5 * n_plots))
        if n_plots == 1:
            axs = [axs]

    else:
        axs = axes
    ax_iter = iter(axs)


    if one:
        #1: grafico solito train e test error
        ax = next(ax_iter)
        zero_train = next((d for d, e in zip(depths_list_float, E_hat_Rs) if e == 0), None)
        ax.plot(depths_list_float, E_hat_Rs, color='#FF007F', marker='o', label=ref_label)
        ax.plot(depths_list_float, E_Rs,     color='#00CED1', marker='o', label=r'$E[R]$')
        if std_hat_Rs is not None:
            ax.fill_between(depths_list_float, E_hat_Rs - std_hat_Rs, E_hat_Rs + std_hat_Rs,
                             color='#FF007F', alpha=0.15, linewidth=0)
        if std_Rs is not None:
            ax.fill_between(depths_list_float, E_Rs - std_Rs, E_Rs + std_Rs,
                             color='#00CED1', alpha=0.15, linewidth=0)
        # logica per if-else above (overft threshold a 0, se computato esplicitamente, non ha senso quindi ignoro il caso)
        if threshold:
            ax.axvline(threshold, color='red',  linestyle='--', linewidth=1.2, label='theoretical overfitting th.')
        # questo è per plottare empirical overft threshold
        # if zero_train is not None:
        #     ax.axvline(zero_train, color='blue', linestyle='--', linewidth=1.2, label='empirical overfitting th.')
        ax.set_ylim(bottom=0)
        ax.set_xlabel('d'); ax.set_ylabel('error')
        ax.legend(); ax.grid(True, alpha=0.3)

    if two:
        #2: grafico che confronta il bound teorico e il gap empirico
        ax = next(ax_iter)
        zero_train = next((d for d, e in zip(depths_list_float, E_hat_Rs) if e == 0), None)
        ax.plot(depths_list_float, E_gaps, color='#32CFFFFF', marker='o', label=gap_label)
        ax.plot(depths_list_float, bound,  color='gray', linestyle='--', label=bound_label)
        if std_gaps is not None:
            ax.fill_between(depths_list_float, E_gaps - std_gaps, E_gaps + std_gaps,
                             color='#32CFFFFF', alpha=0.15, linewidth=0)
        if threshold:
            ax.axvline(threshold, color='red',  linestyle='--', linewidth=1.2, label='theoretical overfitting th.')
        # if zero_train is not None:
        #     ax.axvline(zero_train, color='blue', linestyle='--', linewidth=1.2, label='empirical overfitting th.')
        ax.set_ylim(bottom=0)
        ax.set_xlabel('d'); ax.set_ylabel('gap')
        ax.legend(); ax.grid(True, alpha=0.3)

    if three:
        #3: grafico che confronta la LHS E[R] con la RHS E[hat_R] + bound
        ax = next(ax_iter)
        zero_train = next((d for d, e in zip(depths_list_float, E_hat_Rs) if e == 0), None)
        RHS = E_hat_Rs + bound
        ax.plot(depths_list_float, E_Rs,  color='#FD6428FF', marker='o', label=r'$E[R]$')
        ax.plot(depths_list_float, RHS,   color='#6BC615FF', linestyle='--', marker='s', label=f'{ref_label} + {bound_label}')
        if std_Rs is not None:
            ax.fill_between(depths_list_float, E_Rs - std_Rs, E_Rs + std_Rs,
                             color='#FD6428FF', alpha=0.15, linewidth=0)
        if std_hat_Rs is not None:
            # RHS's randomness comes only from E_hat_Rs (the bound term is deterministic)
            ax.fill_between(depths_list_float, RHS - std_hat_Rs, RHS + std_hat_Rs,
                             color='#6BC615FF', alpha=0.15, linewidth=0)
        if threshold:
            ax.axvline(threshold, color='red',  linestyle='--', linewidth=1.2, label='theoretical overfitting th.')
        # if zero_train is not None:
        #     ax.axvline(zero_train, color='blue', linestyle='--', linewidth=1.2, label='empirical overfitting th.')
        ax.set_ylim(bottom=0)
        ax.set_xlabel('d'); ax.set_ylabel('error')
        ax.legend(); ax.grid(True, alpha=0.3)

    if axes is None:
        plt.tight_layout(rect=[0, 0, 1, 0.96])
        plt.show()

NameError: name 'centered_parabola' is not defined

#### animated plots

In [11]:

def animate_learned_function(model, n, best_pred, best_x, best_y, x_grid,
                             f=centered_parabola, interval=600):
    '''
    build the animation of the learned function across depths from
    already-computed per-depth predictions/samples (e.g. the ones returned by
    main_loop(), or by animate_learned_function() below): for each depth,
    shows the training points that produced the plotted fit and the
    corresponding learned function, alongside sign(f(x)).
    '''
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.plot(x_grid, np.where(f(x_grid) >= 0, 1, -1), color='#0b0b0b', linewidth=1.5, linestyle='--', label=r'sign$(f(x))$')
    scat = ax.scatter([], [], color='#2a78d6', s=30, alpha=0.7, zorder=3, label='training data')
    line, = ax.plot([], [], color='#e34948', linewidth=2, label='learned function')
    ax.set_ylim(-1.5, 1.5)
    ax.set_xlabel('x'); ax.set_ylabel('predicted class')
    ax.legend(loc='center right')
    ax.grid(True, alpha=0.3)

        # overfitting threshold: left as None for models with no threshold
        # logic defined here (e.g. RandomForestClassifier's hybrid measure)
    threshold = None
    if get_model_class(model) == DecisionTreeClassifier:
        threshold = np.log2(n)
    
    elif get_model_class(model) == MLPClassifier:
        # ~3 params per hidden unit with 1D input → threshold at n/3
        threshold = n / 3

    depths_list, _ = get_depths_list(threshold, model)
    is_forest = get_model_class(model) == RandomForestClassifier

    def update(frame):
        line.set_data(x_grid, best_pred[frame])
        scat.set_offsets(np.column_stack([best_x[frame], best_y[frame]]))
        if is_forest:
            depth_val, n_estimators_val = int(depths_list[frame][0]), int(depths_list[frame][1])
            depth_label = 'full' if depth_val == -1 else depth_val
            ax.set_title(f'depth={depth_label}, n_trees={n_estimators_val}')
        else:
            ax.set_title(f'depth = {int(depths_list[frame])}')
        return line, scat

    anim = FuncAnimation(fig, update, frames=len(depths_list), interval=interval)
    plt.close(fig)
    return anim

def animate_tree_splits(model, n, best_pred, best_x, best_y, x_grid,
                         f=centered_parabola, interval=600):
    '''
    visualize how a decision tree splits the 1D input into branches as depth
    grows: for each depth, shades every leaf's region of x_grid by its
    predicted class, draws a dashed vertical line at each split threshold
    between adjacent leaves, and scatters the training points (colored by
    their true label) at their (x, y) position. as depth increases and more
    lines/leaves appear, the training points visually end up sorted into
    thinner and thinner regions — literally the effect of the tree's splits
    partitioning the 50 datapoints into more and more branches.

    only meaningful for DecisionTreeClassifier, the only model here whose
    predictions come from an explicit axis-aligned partition of the input.

    reuses the predictions/samples already computed by main_loop() (passed
    in as best_pred/best_x/best_y/x_grid) instead of refitting: at depth d,
    best_pred[d] is the predicted class along x_grid, and the points where it
    changes are exactly the tree's split thresholds at that depth.
    '''
    assert get_model_class(model) == DecisionTreeClassifier, \
        'animate_tree_splits only supports DecisionTreeClassifier'

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.set_xlim(x_grid.min(), x_grid.max())
    ax.set_ylim(-1.5, 1.5)
    ax.set_xlabel('x'); ax.set_ylabel('y')
    ax.grid(True, alpha=0.3)

    threshold = np.log2(n)
    depths_list, _ = get_depths_list(threshold, model)

    scat = ax.scatter([], [], s=30, zorder=3, edgecolor='k', linewidth=0.4)
    title = ax.set_title('')
    artists = {'regions': [], 'lines': []}

    def update(frame):
        for p in artists['regions']: p.remove()
        for l in artists['lines']: l.remove()
        artists['regions'] = []; artists['lines'] = []

        pred = best_pred[frame]
        # leaf boundaries: midpoints between consecutive x_grid points where
        # the predicted class changes (i.e. where a split separates leaves)
        change_idx = np.where(np.diff(pred) != 0)[0]
        boundaries = ([x_grid[0]]
                       + [(x_grid[i] + x_grid[i + 1]) / 2 for i in change_idx]
                       + [x_grid[-1]])

        for start, end in zip(boundaries[:-1], boundaries[1:]):
            mid_idx = np.searchsorted(x_grid, (start + end) / 2)
            leaf_class = pred[min(mid_idx, len(pred) - 1)]
            color = '#2a78d6' if leaf_class == 1 else '#e34948'
            artists['regions'].append(ax.axvspan(start, end, color=color, alpha=0.12, zorder=0))

        for b in boundaries[1:-1]:
            artists['lines'].append(ax.axvline(b, color='#555', linestyle='--', linewidth=1, zorder=1))

        x, y = best_x[frame], best_y[frame]
        scat.set_offsets(np.column_stack([x, y]))
        scat.set_color(np.where(y == 1, '#2a78d6', '#e34948'))

        n_leaves = len(boundaries) - 1
        title.set_text(f'depth = {int(depths_list[frame])}, leaves = {n_leaves}')
        return [scat, title, *artists['regions'], *artists['lines']]

    anim = FuncAnimation(fig, update, frames=len(depths_list), interval=interval)
    plt.close(fig)
    return anim
